# Zadanie 7.1: PINN dla modelu z terapią (issue #24)

Sieć neuronowa uczy się rozwiązania $u(x, t)$ równania
$\partial_t u = D\,\partial_{xx} u + \rho u (1 - u/K) - \beta R(x) U(t) u$ bez danych z solvera: strata to sama reszta równania
w losowych punktach kolokacji. Solver B służy tylko do oceny.

**Zmienna.** Sieć uczy się $w = \log(u/K)$. Podstawienie $u = K e^w$ daje
$$w_t = D\,(w_{xx} + w_x^2) + \rho\,(1 - e^w) - \beta R(x) U(t).$$
Błąd w $w$ to błąd względny w $u$, więc sieć nie myli się o stałe $10^{-3}$ przed frontem, gdzie $u$ jest o wiele rzędów mniejsze
(w eksperymentach do magisterki taka sieć ucząca się samego $u$ przy $T = 10$ dawała rozwiązanie trywialne $u = 0$).
Człon dawki jest w $w$ addytywny: frakcja obniża $w$ o $\beta R(x)$.

**Postać sieci.** $w(x, t) = w_0(x) + t \cdot 2 \cdot N(x, t, C(t))$, gdzie $C(t) = \int_0^t U$ to liczba frakcji podanych do chwili $t$
(„zegar dawki”). W $t = 0$ warunek początkowy jest spełniony dokładnie, a zegar dawki pozwala sieci zrobić skok przy każdej frakcji.

**Uczenie.** MLP tanh 4 × 64, Adam, 30 tys. iteracji z krokiem malejącym od $2\cdot10^{-3}$ do $2\cdot10^{-5}$.
W każdej iteracji 8192 nowe punkty kolokacji: $x$ równomiernie, $t$ równomiernie albo (25% punktów) blisko impulsów dawki.
Warunków brzegowych nie wymuszamy: brzegi są daleko od guza, gdzie $u \sim e^{-30}$. Szczegóły i próby, które nie zadziałały,
są w opisie modułu `src/offtarget/pinn.py`.

**Scenariusze** (jak w `zad2_02`, dawki ilustracyjne): pole pokrywa guz (duża dawka), pole za wąskie, pole przesunięte.
10 frakcji co 0,4 od $t = 1$, $t \in [0, 10]$, obszar $[-40, 40]$.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from offtarget.config import FIGURES, MODELS, TABLES, seed_everything
from offtarget.model import Params, Schedule, grid
from offtarget.observables import min_mass, time_to_regrowth, tumor_mass
from offtarget.pinn import DEVICE, TrainConfig, log_initial_condition, predict, train_cached
from offtarget.solvers import simulate

rng = seed_everything()
print("urządzenie:", DEVICE)

x, dx = grid(80.0, 800)
t = np.round(np.linspace(0.0, 10.0, 201), 10)
schedule = Schedule.regular(n=10, start=1.0, interval=0.4)
plans = {
    "pole pokrywa guz": Params(beta=1.5, s=0.0, r=20.0),
    "pole za wąskie": Params(beta=1.5, s=0.0, r=3.0),
    "pole przesunięte": Params(beta=1.5, s=6.0, r=8.0),
}

reference, solver_seconds = {}, {}
for name, p in plans.items():
    tic = time.perf_counter()
    reference[name] = simulate(p, schedule, x, t, method="cn")
    solver_seconds[name] = time.perf_counter() - tic

## Uczenie

Co 250 iteracji liczymy błąd względny L2 pola $u$ wobec solvera (na rzadszej siatce), tylko do wykresu: do uczenia sieć go nie widzi.
Wytrenowane sieci zapisują się w `results/models/`; ponowne uruchomienie z tą samą konfiguracją je wczytuje.
Uczenie trwa ok. 4–5 min na scenariusz na GPU Apple (MPS).

In [ ]:
cfg = TrainConfig()
x_mon, t_mon = x[::2], t[::2]


def rel_l2(u, ref):
    return float(np.linalg.norm(u - ref) / np.linalg.norm(ref))


models, histories = {}, {}
for name, p in plans.items():
    ref_mon = reference[name][::2, ::2]
    models[name], histories[name] = train_cached(
        MODELS / f"zad7_01_{name.replace(' ', '_')}.pt", p, schedule, (x[0], x[-1]), t[-1], log_initial_condition(), cfg,
        monitor=lambda m, ref_mon=ref_mon: rel_l2(predict(m, x_mon, t_mon), ref_mon))
    print(f"{name}: {histories[name]['seconds']:.0f} s, błąd na koniec {histories[name]['monitor'][-1]:.3f}")

## Porównanie z solverem

Błąd względny L2 całego pola $u(x, t)$, największy błąd względny masy $M(t)$, najmniejsza masa i czas do odrostu
(pierwsza chwila po minimum, w której $M$ wraca do $M(0)$). Czas odczytu to jedno wywołanie sieci na całej siatce.

In [ ]:
rows, pinn_fields = [], {}
for name in plans:
    tic = time.perf_counter()
    u = predict(models[name], x, t)
    if DEVICE == "mps":
        torch.mps.synchronize()
    read_seconds = time.perf_counter() - tic
    pinn_fields[name] = u
    ref = reference[name]
    M, MP = tumor_mass(ref, x), tumor_mass(u, x)
    rows.append({
        "scenariusz": name,
        "błąd L2 pola u": rel_l2(u, ref),
        "największy błąd względny M(t)": float(np.max(np.abs(MP - M) / M)),
        "M_min / M(0), solver": min_mass(t, M)[0] / M[0],
        "M_min / M(0), PINN": min_mass(t, MP)[0] / MP[0],
        "czas do odrostu, solver": time_to_regrowth(t, M),
        "czas do odrostu, PINN": time_to_regrowth(t, MP),
        "uczenie PINN [s]": float(histories[name]["seconds"]),
        "odczyt PINN [s]": read_seconds,
        "solver B [s]": solver_seconds[name],
    })
table = pd.DataFrame(rows)
table.to_csv(TABLES / "zad7_01_pinn.csv", index=False)
table.style.format(precision=3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for (name, ref), c in zip(reference.items(), plt.cm.tab10.colors):
    M, MP = tumor_mass(ref, x), tumor_mass(pinn_fields[name], x)
    axes[0].plot(t, M / M[0], color=c, lw=2, label=name)
    axes[0].plot(t, MP / MP[0], color=c, lw=1.5, ls="--")
axes[0].plot([], [], "k-", lw=2, label="solver B")
axes[0].plot([], [], "k--", lw=1.5, label="PINN")
axes[0].set(xlabel="czas t", ylabel="M(t) / M(0)", title="Masa guza", ylim=(0, None))
axes[0].legend(fontsize=8)

name = "pole przesunięte"
for k, c in zip([int(np.argmin(np.abs(t - v))) for v in (0.0, 5.0, 10.0)], plt.cm.viridis([0.0, 0.45, 0.85])):
    axes[1].plot(x, reference[name][k], color=c, lw=2, label=f"t = {t[k]:g}")
    axes[1].plot(x, pinn_fields[name][k], color=c, lw=1.5, ls="--")
axes[1].set(xlabel="x", ylabel="gęstość u", title=f"{name}: profile (linia ciągła: solver, przerywana: PINN)", xlim=(-30, 30))
axes[1].legend(fontsize=8)

for name, c in zip(plans, plt.cm.tab10.colors):
    h = histories[name]
    axes[2].semilogy(h["iteration"], h["monitor"], color=c, label=name)
axes[2].set(xlabel="iteracja", ylabel="błąd względny L2 pola u", title="Uczenie")
axes[2].grid(True, which="both", alpha=0.3)
axes[2].legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIGURES / "zad7_01_pinn.png", dpi=150)

## Wnioski

Liczby z przebiegu 19.11.2026 na MacBooku (GPU Apple, MPS), jedno ziarno.

- **PINN odtwarza wszystkie trzy scenariusze.** Błąd względny L2 całego pola $u(x, t)$: 1,1% (pole pokrywa guz), 3,3% (pole za wąskie),
  2,2% (pole przesunięte). Przy odroście masa zgadza się z solverem do 2,8–3,0%, najmniejsza masa do 0,01–0,013 $M(0)$,
  a czas do odrostu różni się o 0,05–0,1 (1–2 kroki zapisu).
- **Pole pokrywa guz:** największy błąd względny masy to 8,9%, ale dotyczy masy rzędu $3\cdot10^{-5}\,M(0)$
  (najmniejsza masa $2{,}7\cdot10^{-5}$ wobec $3{,}0\cdot10^{-5}$). Dzięki zmiennej $w = \log u$ sieć trzyma błąd względny nawet tam,
  gdzie guza prawie nie ma.
- **Gdzie sieć się myli:** w plateau $u$ wychodzi nieco ponad $K$ (do ok. 1,02), bo postać $K e^w$ tego nie zabrania,
  a mały guzek odrastający za prawym brzegiem pola przesuniętego ma w $t = 10$ gęstość 0,40 zamiast 0,44.
  To akurat mechanizm, o który chodzi w temacie (komórki spoza pola), więc tam trzeba patrzeć przy ulepszaniu.
- **Co zdecydowało o wyniku:** zmienna logarytmiczna, zegar dawki $C(t)$ na wejściu sieci, nowe punkty kolokacji w każdej iteracji
  z zagęszczeniem przy impulsach i 30 tys. iteracji (przy 10 tys. błąd ok. 10%). Dawka wpisana wprost w postać sieci dała 86% błędu
  (opis w `src/offtarget/pinn.py`).
- **Koszt:** uczenie ok. 4,4 min na scenariusz, solver B liczy cały przebieg w 0,03 s. Odczyt wytrenowanej sieci na tej samej siatce
  trwa ok. 0,01 s (pierwsze wywołanie 0,4 s, bo rozgrzewa GPU), więc jest tylko ok. 3 razy szybszy od solvera. Sieć uczona osobno dla
  każdego scenariusza się nie opłaca. Opłaci się dopiero sieć, która dostaje parametry planu (r, s, β) na wejściu albo rozwiązuje
  zadanie odwrotne (#36, #42, #44).
- **Czego nie sprawdziliśmy:** rozrzutu między ziarnami, klinicznego harmonogramu z wąskimi impulsami (`clinical_schedule()` z #10)
  i porównania z siecią uczącą się samego $u$ na tych scenariuszach.